# 3. Extract 2D Obstacles

Extracts 2D obstacle polygons from the BGT-labeled point clouds produced by **step 2.5**.

Each tile is voxelized, height-above-ground is computed, and connected components above
the height threshold are clustered into concave polygons.

**Input:** `bgt_labeled_{tilecode}.laz`  
**Output:** `obstacles_labeled_{tilecode}.geojson`

**Run this after step 2.5 (BGT Object Labeling).**

## Config

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('.').resolve()))

from config import LABELED_DIR, OBSTACLES_DIR, SETUP_TILECODES

tilecodes = SETUP_TILECODES

DIR_IN = str(LABELED_DIR / "bgt_labeled_")
DIR_OUT = OBSTACLES_DIR

CRS = 'EPSG:28992'
MAX_AREA = 10  # m² — drop oversized clusters (buildings, ground patches)

DIR_OUT.mkdir(parents=True, exist_ok=True)

## Imports

In [ ]:
import numpy as np
import utils.obstacles_utils as obstacles_utils

## Extract clusters

In [ ]:
clusters_dict = {}

for tilecode in tilecodes:
    laz_file = f"{DIR_IN}{tilecode}.laz"
    print(f"\n{tilecode}")
    clusters = obstacles_utils.cluster_obstacles(laz_file)
    clusters_dict[tilecode] = clusters
    print(f"  Detected obstacles: {len(clusters)}")

## Save to GeoJSON

In [ ]:
for tilecode, clusters in clusters_dict.items():
    gdf = obstacles_utils.clusters_to_concave_polygons(clusters, CRS)
    gdf['tilecode'] = tilecode
    gdf['area'] = gdf.area
    gdf = gdf[gdf.area < MAX_AREA]

    out_file = DIR_OUT / f"obstacles_labeled_{tilecode}.geojson"
    gdf.to_file(out_file, driver='GeoJSON')
    print(f"{tilecode}: {len(gdf)} obstacles -> {out_file}")

## Optional: plot last tile

In [ ]:
gdf.plot(edgecolor='red', facecolor='none', figsize=(8, 8))